In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import bitsandbytes
from PIL import Image
from IPython.display import display
from pathlib import Path
import sys
import torch
import open_clip
import chromadb
import re
import os
import numpy as np
import subprocess
from PIL import Image
from pocket_tts import TTSModel
import scipy.io.wavfile
from scipy.io import wavfile
import soundfile as sf

from sqlalchemy.orm import Session
from sqlalchemy import text

In [3]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine

In [4]:
!nvidia-smi

Wed Mar  4 14:45:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8             15W /   85W |     156MiB /   6144MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [40]:
script_path = r'C:\Uni\GP\ECHO\notebooks\video_generation\video_generation_pharaohs\Scripts\Ramesses II.txt'
with open(script_path, 'r', encoding='utf-8') as file:
    script = file.read()

paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.',
 'Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.',
 "He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers."

In [41]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in paragraphs:
    audio = tts_model.generate_audio(voice_state, p)
    scipy.io.wavfile.write(f"Outputs\\output_{i}.wav", tts_model.sample_rate, audio.numpy())
    i += 1

In [42]:
wav_files = sorted([f for f in os.listdir("Outputs") if f.endswith('.wav')])
print(wav_files)

audio_data = []
samplerate = None

for file_name in wav_files:
    file_name = os.path.join("Outputs", file_name)
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write("Outputs\\Rameses II.wav", combined, samplerate)

['Rameses II.wav', 'output_0.wav', 'output_1.wav', 'output_2.wav', 'output_3.wav']


In [43]:
durations = []
images_needed = []
seconds = []
for i in range(len(paragraphs)):
    file_path = f"Outputs\\output_{i}.wav"
    # Returns the sample rate (Fs) and the data as a NumPy array
    Fs, data = wavfile.read(file_path) 
    # The length of the data array divided by the sample rate gives the duration
    duration_seconds = len(data) / float(Fs)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)
    os.remove(file_path)

print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Seconds for each image: {seconds}")

Durations of audio files: [24.96, 19.52, 15.84, 21.52]
Images needed for each paragraph: [4, 3, 2, 3]
Seconds for each image: [6.24, 6.24, 6.24, 6.24, 6.506666666666667, 6.506666666666667, 6.506666666666667, 7.92, 7.92, 7.173333333333333, 7.173333333333333, 7.173333333333333]


In [ ]:
i = 0
image_text_chunks = []
for p in paragraphs:
    sentences = re.split(r'(?<=[.!?]) +', p)
    sentences = [s.strip() for s in sentences if s.strip()]
    images_for_paragraph = images_needed[i]

    images_for_paragraph = min(images_for_paragraph, len(sentences))

    total_sentences = len(sentences)

    base = total_sentences // images_for_paragraph
    remainder = total_sentences % images_for_paragraph

    groups = []
    start = 0

    for img_idx in range(images_for_paragraph):
        extra = 1 if img_idx < remainder else 0
        end = start + base + extra

        chunk = " ".join(sentences[start:end])
        groups.append(chunk)

        start = end

    image_text_chunks.append(groups)
        
    i += 1
    print(f"Paragraph {i} split into {len(groups)} image text chunks.")
    print("-" * 40)
    print(groups)

Paragraph 1 split into 4 image text chunks.
----------------------------------------
4
['Rameses II was one of ancient Egypt’s greatest pharaohs.', 'Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.', 'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.', 'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.']
Paragraph 2 split into 3 image text chunks.
----------------------------------------
3
['Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.', 'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.', 'Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.']
Paragraph 3 split into 2 image text chunks.
----------------------------------------
2
["He continue

In [ ]:
for paragraph_chunks in image_text_chunks:
    for chunk in paragraph_chunks:
        text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
    
    with torch.no_grad():
        text_embedding = model.encode_text(text_tokens)
        text_embedding /= text_embedding.norm(dim=-1, keepdim=True)
    
    
        

In [30]:
name = Path(script_path).stem
with engine.connect() as conn:
    result = conn.execute(text("SELECT image_embedding FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = :name"), {"name": name})

    pharaoh_images = result.fetchall()

pharaoh_images

2026-03-04 14:57:24,406 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-04 14:57:24,408 INFO sqlalchemy.engine.Engine SELECT image_embedding FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = %(name)s


INFO:sqlalchemy.engine.Engine:SELECT image_embedding FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = %(name)s


2026-03-04 14:57:24,408 INFO sqlalchemy.engine.Engine [generated in 0.00235s] {'name': 'Ramesses II'}


INFO:sqlalchemy.engine.Engine:[generated in 0.00235s] {'name': 'Ramesses II'}


2026-03-04 14:57:24,426 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


[('[0.013862469,-0.012793625,-0.014664471,-0.043693576,0.044361103,0.008973969,-0.015818063,0.010515705,-0.0137487,-0.017900478,-0.011147908,0.019887142 ... (12584 characters truncated) ... 016267668,0.0042939293,-0.023664553,-0.030056803,0.004420877,-0.057095546,0.0023231546,0.022635348,-0.033229537,-0.054128304,0.030762209,0.007595442]',),
 ('[-0.01680345,-0.026534947,-0.009838806,-0.012731712,0.005601219,0.018987896,-0.073591,0.005853427,-0.00013216161,-0.0051507195,-0.012175544,0.0516231 ... (12544 characters truncated) ... .029635068,0.017442463,-0.02734546,-0.039532907,-0.008919614,-0.045394953,0.008707426,-0.00083306176,0.0013956558,-0.01217285,0.015211243,-0.0320484]',),
 ('[0.00031804602,0.029013593,0.01500648,-0.01806565,0.0065373774,0.0012895663,-0.038069915,0.016310679,-0.023318578,0.0020659908,-0.014946164,0.0237293 ... (12476 characters truncated) ... -0.019262876,-0.012532488,0.0018017347,-0.035231918,-0.020377986,-0.06968061,-0.04424726,0.01895878,0.0043099504,0.0091353